In [19]:
# ============================================================
# WIN MODEL (P(win)) — 2025 walk-forward + evaluation:
#   "higher win% guys finished relatively high (Top10)"
#
# Uses:
#   - combined_rounds_all_2017_2026.csv  (all tours)
#   - Odds_and_Results.xlsx              (field + finish_num + close_odds)
#   - OAD_2024/2025/2026.xlsx            (event schedule + event_name + start_date)
#
# Key choices:
#   - No future info: features use rounds <= (event_start_date - 1 day)
#   - Primary driver: rolling sg_total (L40/L24/L12)
#   - Odds: included as a SMALL feature (scaled), not allowed to dominate
#   - Model: regularized LogisticRegression (stable)
#   - Output: per-event Top10 board + summary metrics
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

# -----------------------------
# PATHS (EDIT IF NEEDED)
# -----------------------------
BASE_DIR = "/Users/joshmacbook/python_projects/OAD/Data/in Use"
ROUNDS_PATH = f"{BASE_DIR}/combined_rounds_all_2017_2026.csv"
ODDS_XLSX   = f"{BASE_DIR}/Odds_and_Results.xlsx"
SCHED_2024  = f"{BASE_DIR}/OAD_2024.xlsx"
SCHED_2025  = f"{BASE_DIR}/OAD_2025.xlsx"
SCHED_2026  = f"{BASE_DIR}/OAD_2026.xlsx"

# -----------------------------
# SETTINGS (EDIT)
# -----------------------------
BACKTEST_YEAR = 2025
WINDOWS = (40, 24, 12)

LR_C = 0.05
LR_MAX_ITER = 2000

ODDS_SCALE = 0.15       # keep small (you found 0.10-ish works well)
REL_HIGH_FINISH = 10    # Top10
TOP_GROUP_N = 10
TOPK_LIST = [1, 3, 5, 10, 20]

# choose one feature set
FEAT_SET_KEY = "baseline_plus_layoff"

# -----------------------------
# HELPERS
# -----------------------------
def safe_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if np.unique(y_true).size < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)

def _spearman_rank_corr(a, b):
    s = pd.DataFrame({"a": a, "b": b}).dropna()
    if len(s) < 3:
        return np.nan
    ra = s["a"].rank(method="average")
    rb = s["b"].rank(method="average")
    return float(ra.corr(rb, method="pearson"))

def _logit(p, eps=1e-8):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def _to_naive_datetime(s: pd.Series) -> pd.Series:
    """Force datetime64[ns] naive for merge_asof stability (kills tz/object weirdness)."""
    s = pd.to_datetime(s, errors="coerce")
    try:
        if hasattr(s.dt, "tz") and s.dt.tz is not None:
            s = s.dt.tz_convert(None)
    except Exception:
        pass
    return s

# ============================================================
# 1) LOAD SCHEDULES -> sched_all with standard cols:
#    year, event_id, event_start_date, event_name
# ============================================================
sched_24 = pd.read_excel(SCHED_2024)
sched_25 = pd.read_excel(SCHED_2025)
sched_26 = pd.read_excel(SCHED_2026)

sched_all_raw = pd.concat([sched_24, sched_25, sched_26], ignore_index=True)

# Find date column
date_col = None
for cand in ["start_date", "event_start_date", "date", "event_date"]:
    if cand in sched_all_raw.columns:
        date_col = cand
        break
if date_col is None:
    raise ValueError("Schedule missing start date column (start_date / event_start_date / date / event_date).")

# Find name column
name_col = None
for cand in ["event_name", "tournament", "name"]:
    if cand in sched_all_raw.columns:
        name_col = cand
        break
if name_col is None:
    raise ValueError("Schedule missing event name column (event_name / tournament / name).")

sched_all = sched_all_raw.copy()
sched_all["year"] = pd.to_numeric(sched_all["year"], errors="coerce")
sched_all["event_id"] = pd.to_numeric(sched_all["event_id"], errors="coerce")
sched_all[date_col] = _to_naive_datetime(sched_all[date_col])
sched_all[name_col] = sched_all[name_col].astype(str)

sched_all = sched_all.dropna(subset=["year", "event_id", date_col]).copy()
sched_all["year"] = sched_all["year"].astype(int)
sched_all["event_id"] = sched_all["event_id"].astype(int)

sched_all = sched_all.rename(columns={date_col: "event_start_date", name_col: "event_name"})
sched_all = sched_all[["year","event_id","event_start_date","event_name"]].copy()

# backtest events (2025)
sched_bt = (
    sched_all[sched_all["year"] == BACKTEST_YEAR]
    .dropna(subset=["event_id","event_start_date"])
    .sort_values("event_start_date")
    .reset_index(drop=True)
)

print("Backtest events:", len(sched_bt))
display(sched_bt[["event_id","event_start_date","event_name"]].head(25))

# ============================================================
# 2) LOAD ODDS/RESULTS
# ============================================================
odds = pd.read_excel(ODDS_XLSX)

need_odds = {"year","event_id","dg_id","close_odds","finish_num"}
missing = need_odds - set(odds.columns)
if missing:
    raise ValueError(f"Odds_and_Results.xlsx missing required columns: {sorted(missing)}")

for c in ["year", "event_id", "dg_id"]:
    odds[c] = pd.to_numeric(odds[c], errors="coerce")

odds["close_odds"] = pd.to_numeric(odds["close_odds"], errors="coerce")
odds["finish_num"] = pd.to_numeric(odds["finish_num"], errors="coerce")

odds = odds.dropna(subset=["year","event_id","dg_id"]).copy()
odds["year"] = odds["year"].astype(int)
odds["event_id"] = odds["event_id"].astype(int)
odds["dg_id"] = odds["dg_id"].astype(int)

# Restrict odds to event_ids that appear in schedule
oad_event_ids = sorted(sched_all["event_id"].unique().tolist())
odds = odds[odds["event_id"].isin(oad_event_ids)].copy()

# ============================================================
# 3) LOAD ROUNDS + BUILD ROLLING SG_TOTAL FEATURES
# ============================================================
rounds = pd.read_csv(ROUNDS_PATH)

need_rounds = {"round_date","dg_id","player_name","sg_total"}
missing = need_rounds - set(rounds.columns)
if missing:
    raise ValueError(f"combined_rounds file missing required columns: {sorted(missing)}")

rounds["round_date"] = _to_naive_datetime(rounds["round_date"])
rounds["dg_id"] = pd.to_numeric(rounds["dg_id"], errors="coerce")
rounds["sg_total"] = pd.to_numeric(rounds["sg_total"], errors="coerce")
rounds["player_name"] = rounds["player_name"].astype(str)

rounds = rounds.dropna(subset=["round_date","dg_id","sg_total"]).copy()
rounds["dg_id"] = rounds["dg_id"].astype(int)

# stable name map from latest seen name per dg_id
name_map = (
    rounds.dropna(subset=["dg_id","player_name"])
          .sort_values(["dg_id","round_date"])
          .groupby("dg_id")["player_name"]
          .last()
          .to_dict()
)

rounds = rounds.sort_values(["dg_id","round_date"]).reset_index(drop=True)

def build_player_rolling_features(rounds_df: pd.DataFrame, windows=(40,24,12)) -> pd.DataFrame:
    df = rounds_df[["dg_id","round_date","sg_total"]].copy()
    df = df.sort_values(["dg_id","round_date"]).reset_index(drop=True)

    g = df.groupby("dg_id", group_keys=False)
    df["n_rounds_to_date"] = g.cumcount() + 1

    for w in windows:
        df[f"sg_total_L{w}"] = (
            g["sg_total"]
            .rolling(window=w, min_periods=min(10, w))
            .mean()
            .reset_index(level=0, drop=True)
        )
    return df

rounds_roll = build_player_rolling_features(rounds, windows=WINDOWS)

# IMPORTANT: merge_asof requires right "on" key sorted globally
rounds_roll = rounds_roll.dropna(subset=["round_date","dg_id"]).copy()
rounds_roll["round_date"] = _to_naive_datetime(rounds_roll["round_date"])
rounds_roll["dg_id"] = pd.to_numeric(rounds_roll["dg_id"], errors="coerce")
rounds_roll = rounds_roll.dropna(subset=["round_date","dg_id"]).copy()
rounds_roll["dg_id"] = rounds_roll["dg_id"].astype(int)
rounds_roll = rounds_roll.sort_values(["round_date","dg_id"]).reset_index(drop=True)

# ============================================================
# 4) BUILD MODELING TABLE X (player-event rows) with cutoff = start-1
# ============================================================
evs = sched_all.copy()
evs["cutoff_date"] = evs["event_start_date"] - pd.Timedelta(days=1)

# player-event table (from odds/results) + event meta
pe = odds.merge(
    evs[["year","event_id","event_start_date","cutoff_date","event_name"]],
    on=["year","event_id"],
    how="inner",
    suffixes=("", "_sched"),
).copy()

# ensure pe has a clean event_name
if "event_name_sched" in pe.columns:
    pe["event_name"] = pe["event_name_sched"].fillna(pe.get("event_name"))
pe["event_name"] = pe["event_name"].fillna("UNKNOWN_EVENT").astype(str)

# ensure player_name exists
pe["player_name"] = pe["dg_id"].map(name_map).fillna(pe["dg_id"].apply(lambda x: f"dg_{x}"))

# label
pe["y_win"] = (pe["finish_num"] == 1).astype(int)

# -----------------------------
# ASOF MERGE (robust)
#   Must be globally sorted by ON key, not by dg_id first.
# -----------------------------
left = pe.copy()
right = rounds_roll.copy()

left["cutoff_date"] = _to_naive_datetime(left["cutoff_date"])
right["round_date"] = _to_naive_datetime(right["round_date"])

left["dg_id"] = pd.to_numeric(left["dg_id"], errors="coerce")
right["dg_id"] = pd.to_numeric(right["dg_id"], errors="coerce")

left = left.dropna(subset=["dg_id","cutoff_date"]).copy()
right = right.dropna(subset=["dg_id","round_date"]).copy()

left["dg_id"] = left["dg_id"].astype(int)
right["dg_id"] = right["dg_id"].astype(int)

# CRITICAL: sort by ON then BY
left  = left.sort_values(["cutoff_date","dg_id"]).reset_index(drop=True)
right = right.sort_values(["round_date","dg_id"]).reset_index(drop=True)

try:
    X = pd.merge_asof(
        left,
        right,
        by="dg_id",
        left_on="cutoff_date",
        right_on="round_date",
        direction="backward",
        allow_exact_matches=True,
    )
except ValueError as e:
    # fallback per-player (slower, but guaranteed)
    print("merge_asof failed; falling back per dg_id. Error:", e)
    parts = []
    right_g = right.sort_values(["dg_id","round_date"])
    left_g  = left.sort_values(["dg_id","cutoff_date"])
    for pid, lsub in left_g.groupby("dg_id", sort=False):
        rsub = right_g[right_g["dg_id"] == pid]
        if rsub.empty:
            lsub2 = lsub.copy()
            for c in right.columns:
                if c not in lsub2.columns:
                    lsub2[c] = np.nan
            parts.append(lsub2)
        else:
            merged = pd.merge_asof(
                lsub.sort_values("cutoff_date"),
                rsub.sort_values("round_date"),
                left_on="cutoff_date",
                right_on="round_date",
                direction="backward",
                allow_exact_matches=True,
            )
            parts.append(merged)
    X = pd.concat(parts, ignore_index=True)

# keep last matched round date so we can compute inactivity
X = X.rename(columns={"round_date": "last_round_date"})

# ensure X has event_name (belt + suspenders)
if "event_name" not in X.columns:
    for cand in ["event_name_sched", "event_name_x", "event_name_y"]:
        if cand in X.columns:
            X["event_name"] = X[cand]
            break
    if "event_name" not in X.columns:
        X["event_name"] = "UNKNOWN_EVENT"

# ensure player_name exists in X (belt + suspenders)
if "player_name" not in X.columns:
    X["player_name"] = X["dg_id"].map(name_map).fillna(X["dg_id"].apply(lambda x: f"dg_{x}"))

# -----------------------------
# Derived features
# -----------------------------
X["days_since_last_round"] = (pd.to_datetime(X["cutoff_date"]) - pd.to_datetime(X["last_round_date"])).dt.days
X["days_since_last_round"] = pd.to_numeric(X["days_since_last_round"], errors="coerce").clip(lower=0)
X["days_since_last_round_log1p"] = np.log1p(X["days_since_last_round"].fillna(0.0))

# recency shape
X["sg_total_L12_minus_L40"] = X["sg_total_L12"] - X["sg_total_L40"]
X["sg_total_L24_minus_L40"] = X["sg_total_L24"] - X["sg_total_L40"]

# odds feature: implied prob share within event (logit, scaled)
X["p_odds_raw"] = 1.0 / pd.to_numeric(X["close_odds"], errors="coerce")
X["p_odds_raw"] = X["p_odds_raw"].replace([np.inf, -np.inf], np.nan)
X["p_odds_raw"] = X["p_odds_raw"].fillna(X["p_odds_raw"].median())

X["p_odds_share"] = X["p_odds_raw"] / (X.groupby(["year","event_id"])["p_odds_raw"].transform("sum") + 1e-12)
X["odds_logit_scaled"] = ODDS_SCALE * _logit(X["p_odds_share"].values)
# cap odds share to avoid extreme logits
X["p_odds_share_capped"] = X["p_odds_share"].clip(0.0005, 0.20)
X["odds_logit_scaled"] = ODDS_SCALE * _logit(X["p_odds_share_capped"].values)

# Feature sets
BASE_FEATS  = [f"sg_total_L{w}" for w in WINDOWS] + ["odds_logit_scaled"]
TREND_FEATS = ["sg_total_L12_minus_L40", "sg_total_L24_minus_L40"]
LAYOFF_FEATS = ["days_since_last_round_log1p"]

FEAT_SETS = {
    "baseline": BASE_FEATS,
    "baseline_plus_trend": BASE_FEATS + TREND_FEATS,
    "baseline_plus_layoff": BASE_FEATS + LAYOFF_FEATS,
    "baseline_plus_both": BASE_FEATS + TREND_FEATS + LAYOFF_FEATS,
}

if FEAT_SET_KEY not in FEAT_SETS:
    raise ValueError(f"Unknown FEAT_SET_KEY={FEAT_SET_KEY}. Choose from {list(FEAT_SETS.keys())}")

FEATS = FEAT_SETS[FEAT_SET_KEY]

# require rolling sg features + the chosen feature set + label
X = X.dropna(subset=[f"sg_total_L{w}" for w in WINDOWS]).copy()
X = X.dropna(subset=FEATS + ["y_win", "finish_num", "event_start_date", "event_id", "year"]).copy()

# make sure types are clean
X["year"] = pd.to_numeric(X["year"], errors="coerce").astype(int)
X["event_id"] = pd.to_numeric(X["event_id"], errors="coerce").astype(int)
X["event_start_date"] = _to_naive_datetime(X["event_start_date"])

print("X rows:", len(X), "| years:", sorted(X["year"].unique().tolist())[:10], "| feat_set:", FEAT_SET_KEY)
display(X[["year","event_id","event_name","dg_id","player_name","finish_num","y_win"] + FEATS].head(10))

# ============================================================
# 5) WALK-FORWARD BACKTEST (2025)
# ============================================================
def fit_lr(train_df: pd.DataFrame, feat_cols, label_col="y_win"):
    tr = train_df.dropna(subset=feat_cols + [label_col]).copy()
    ytr = tr[label_col].astype(int).values
    if np.unique(ytr).size < 2:
        return None
    m = LogisticRegression(C=LR_C, max_iter=LR_MAX_ITER, solver="lbfgs")
    m.fit(tr[feat_cols].values, ytr)
    return m

event_boards = []
event_summ = []

bt_events = sched_bt.copy()
print("2025 events to score:", len(bt_events))

for _, ev in bt_events.iterrows():
    ev_id = int(ev["event_id"])
    ev_date = pd.to_datetime(ev["event_start_date"])
    ev_name = str(ev["event_name"])

    train_df = X[
        (X["year"] < BACKTEST_YEAR) |
        ((X["year"] == BACKTEST_YEAR) & (X["event_start_date"] < ev_date))
    ].copy()

    test_df = X[(X["year"] == BACKTEST_YEAR) & (X["event_id"] == ev_id)].copy()

    if train_df.empty or test_df.empty:
        continue

    model = fit_lr(train_df, FEATS, label_col="y_win")
    if model is None:
        continue

    p = model.predict_proba(test_df[FEATS].values)[:, 1]
    test_df = test_df.copy()
    test_df["p_win_raw"] = p
    test_df["p_win_share"] = test_df["p_win_raw"] / (test_df["p_win_raw"].sum() + 1e-12)

    board = test_df[[
        "year","event_id","event_start_date","event_name",
        "dg_id","player_name","finish_num","y_win",
        "p_win_raw","p_win_share",
        "sg_total_L40","sg_total_L24","sg_total_L12",
        "close_odds","p_odds_share"
    ]].copy()

    board = board.sort_values("p_win_share", ascending=False).reset_index(drop=True)
    board["pred_rank"] = np.arange(1, len(board) + 1)

    eval_board = board.dropna(subset=["finish_num"]).copy()

    winner_rows = eval_board[eval_board["finish_num"] == 1]
    winner_rank = int(winner_rows["pred_rank"].iloc[0]) if not winner_rows.empty else np.nan

    spearman = _spearman_rank_corr(eval_board["pred_rank"], eval_board["finish_num"])

    top_group = eval_board.head(TOP_GROUP_N).copy()
    top_group_mean_finish = float(top_group["finish_num"].mean()) if len(top_group) else np.nan
    top_group_median_finish = float(top_group["finish_num"].median()) if len(top_group) else np.nan
    top_group_top10_rate = float((top_group["finish_num"] <= REL_HIGH_FINISH).mean()) if len(top_group) else np.nan

    win_in_topk = {f"winner_in_top{K}": (winner_rank <= K) if np.isfinite(winner_rank) else np.nan for K in TOPK_LIST}

    y_true = eval_board["y_win"].values
    y_prob = eval_board["p_win_raw"].values
    brier = brier_score_loss(y_true, y_prob)
    auc = safe_auc(y_true, y_prob)


    event_summ.append({
        "year": BACKTEST_YEAR,
        "event_id": ev_id,
        "event_name": ev_name,
        "event_date": ev_date,
        "n_field": int(len(board)),
        "winner_pred_rank": winner_rank,
        "spearman_predrank_vs_finish": spearman,
        "top_group_n": TOP_GROUP_N,
        "top_group_mean_finish": top_group_mean_finish,
        "top_group_median_finish": top_group_median_finish,
        f"top_group_top{REL_HIGH_FINISH}_rate": top_group_top10_rate,
        "brier": float(brier),
        "auc": float(auc) if np.isfinite(auc) else np.nan,
        **win_in_topk,
    })

    event_boards.append(board)

    top10 = board.head(10).copy()
    top10["p_win_pct"] = (100 * top10["p_win_share"]).round(2)
    top10["p_odds_pct"] = (100 * top10["p_odds_share"]).round(2)

    print(f"\n=== {ev_date.date()} | event_id={ev_id} | {ev_name} ===")
    display(top10[[
        "pred_rank","player_name","dg_id",
        "p_win_pct","p_odds_pct","close_odds",
        "sg_total_L40","sg_total_L24","sg_total_L12",
        "finish_num"
    ]])

boards_2025 = pd.concat(event_boards, ignore_index=True) if event_boards else pd.DataFrame()
summ_2025 = pd.DataFrame(event_summ).sort_values("event_date").reset_index(drop=True)

print("\nEvents scored:", summ_2025["event_id"].nunique() if not summ_2025.empty else 0)
display(summ_2025.head(50))

# ============================================================
# 6) AGGREGATE RESULTS
# ============================================================
if not summ_2025.empty:
    agg = {
        "events": int(summ_2025["event_id"].nunique()),
        "avg_spearman_predrank_vs_finish": float(summ_2025["spearman_predrank_vs_finish"].mean()),
        f"avg_top{REL_HIGH_FINISH}_rate_in_top{TOP_GROUP_N}":
            float(summ_2025[f"top_group_top{REL_HIGH_FINISH}_rate"].mean()),
        "avg_top_group_mean_finish": float(summ_2025["top_group_mean_finish"].mean()),
        "avg_winner_pred_rank": float(summ_2025["winner_pred_rank"].mean()),
        "avg_brier": float(summ_2025["brier"].mean()),
        "avg_auc": float(summ_2025["auc"].mean()),
        "feat_set": FEAT_SET_KEY,
        "lr_c": LR_C,
        "odds_scale": ODDS_SCALE,
    }
    for K in TOPK_LIST:
        col = f"winner_in_top{K}"
        agg[col] = float(pd.to_numeric(summ_2025[col], errors="coerce").mean())
    display(pd.DataFrame([agg]))

# ============================================================
# 7) OPTIONAL: PLAYER SUMMARY ACROSS 2025
# ============================================================
if not boards_2025.empty:
    player_2025 = (
        boards_2025.groupby(["dg_id","player_name"], as_index=False)
        .agg(
            starts=("event_id","nunique"),
            wins=("y_win","sum"),
            avg_pwin_share=("p_win_share","mean"),
            max_pwin_share=("p_win_share","max"),
            avg_pred_rank=("pred_rank","mean"),
        )
        .sort_values(["wins","avg_pwin_share","starts"], ascending=[False, False, False])
        .reset_index(drop=True)
    )
    display(player_2025.head(50))

print("\nDone.")
print("To try trend/layoff features, only change FEAT_SET_KEY at the top.")


Backtest events: 31


,event_id,event_start_date,event_name
0,6,2025-01-12,Sony Open in Hawaii
1,2,2025-01-19,The American Express
2,4,2025-01-25,Farmers Insurance Open
3,5,2025-02-02,AT&T Pebble Beach Pro-Am
4,3,2025-02-09,WM Phoenix Open
5,7,2025-02-16,Genesis Invitational
6,540,2025-02-23,Mexico Open
7,10,2025-03-02,Cognizant Classic
8,9,2025-03-09,Arnold Palmer Invitational
9,11,2025-03-16,The Players Championship


/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_48884/1857285996.py:163: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  rounds = pd.read_csv(ROUNDS_PATH)


X rows: 4410 | years: [2024, 2025, 2026] | feat_set: baseline_plus_layoff


,year,event_id,event_name,dg_id,player_name,finish_num,y_win,sg_total_L40,sg_total_L24,sg_total_L12,odds_logit_scaled,days_since_last_round_log1p
0,2024,6,Sony Open in Hawaii,5665,"Cink, Stewart",24.0,0,0.440450,0.670833,1.082750,-0.854154,0.0
1,2024,6,Sony Open in Hawaii,5768,"Hoffman, Charley",42.0,0,0.089275,0.217375,0.499417,-0.914892,0.0
2,2024,6,Sony Open in Hawaii,6093,"Rose, Justin",57.0,0,0.382300,-0.179958,0.406417,-0.570962,0.0
8,2024,6,Sony Open in Hawaii,7960,"List, Luke",66.0,0,1.069700,1.030375,0.981667,-0.717076,0.0
13,2024,6,Sony Open in Hawaii,8825,"Harman, Brian",18.0,0,1.421575,0.800875,1.248500,-0.495409,0.0
16,2024,6,Sony Open in Hawaii,10104,"Power, Seamus",74.0,0,-0.299025,-0.964542,-1.964917,-0.811083,0.0
17,2024,6,Sony Open in Hawaii,10419,"Noren, Alex",42.0,0,1.116475,1.342375,2.162000,-0.646940,0.0
21,2024,6,Sony Open in Hawaii,11049,"Simpson, Webb",66.0,0,-0.223100,-0.223292,0.067083,-0.811083,0.0
22,2024,6,Sony Open in Hawaii,11276,"Horschel, Billy",18.0,0,0.984525,1.559292,1.119583,-0.707435,0.0
24,2024,6,Sony Open in Hawaii,12359,"Stallings, Scott",42.0,0,-0.346675,0.208250,0.531083,-0.887576,0.0


...


Events scored: 31


,year,event_id,event_name,event_date,n_field,winner_pred_rank,spearman_predrank_vs_finish,top_group_n,top_group_mean_finish,top_group_median_finish,top_group_top10_rate,brier,auc,winner_in_top1,winner_in_top3,winner_in_top5,winner_in_top10,winner_in_top20
0,2025,6,Sony Open in Hawaii,2025-01-12,75,57,0.292464,10,28.4,29.0,0.3,0.013520,0.243243,False,False,False,False,False
1,2025,2,The American Express,2025-01-19,69,6,0.259595,10,18.7,16.5,0.4,0.013935,0.926471,False,False,False,True,True
2,2025,4,Farmers Insurance Open,2025-01-25,69,16,0.210711,10,25.7,15.0,0.3,0.014241,0.779412,False,False,False,False,True
3,2025,5,AT&T Pebble Beach Pro-Am,2025-02-02,78,1,0.465593,10,15.1,8.0,0.6,0.009823,1.000000,True,True,True,True,True
4,2025,3,WM Phoenix Open,2025-02-09,77,6,0.424371,10,18.0,13.5,0.3,0.012631,0.934211,False,False,False,True,True
5,2025,7,Genesis Invitational,2025-02-16,54,22,0.301036,10,17.3,12.5,0.4,0.018612,0.603774,False,False,False,False,False
6,2025,540,Mexico Open,2025-02-23,72,5,0.498744,10,16.5,11.5,0.5,0.013436,0.943662,False,False,True,True,True
7,2025,10,Cognizant Classic,2025-03-02,67,31,0.352346,10,18.1,14.5,0.4,0.014818,0.545455,False,False,False,False,False
8,2025,9,Arnold Palmer Invitational,2025-03-09,51,5,0.466516,10,10.7,9.0,0.5,0.018791,0.920000,False,False,True,True,True
9,2025,11,The Players Championship,2025-03-16,71,1,0.652720,10,13.7,14.0,0.3,0.012488,1.000000,True,True,True,True,True


,events,avg_spearman_predrank_vs_finish,avg_top10_rate_in_top10,avg_top_group_mean_finish,avg_winner_pred_rank,avg_brier,avg_auc,feat_set,lr_c,odds_scale,winner_in_top1,winner_in_top3,winner_in_top5,winner_in_top10,winner_in_top20
0,31,0.453905,0.416129,18.364516,11.741935,0.013123,0.843643,baseline_plus_layoff,0.05,0.15,0.290323,0.419355,0.580645,0.709677,0.774194


,dg_id,player_name,starts,wins,avg_pwin_share,max_pwin_share,avg_pred_rank
0,18417,"Scheffler, Scottie",18,5,0.165783,0.344586,1.333333
1,10091,"McIlroy, Rory",13,3,0.080617,0.154324,6.461538
2,17511,"Straka, Sepp",15,2,0.029059,0.060621,13.200000
3,18628,"Campbell, Brian",9,2,0.010807,0.042519,43.000000
4,27774,"Gotterup, Chris",12,1,0.039925,0.169041,14.166667
5,14139,"Thomas, Justin",16,1,0.029941,0.068981,11.687500
6,24968,"Griffin, Ben",20,1,0.029535,0.074603,16.900000
7,14578,"Henley, Russell",14,1,0.025529,0.041689,15.928571
8,21891,"Kitayama, Kurt",13,1,0.021886,0.049497,22.846154
9,17536,"Spaun, J.J.",18,1,0.018891,0.039881,18.555556



Done.
To try trend/layoff features, only change FEAT_SET_KEY at the top.


### ------------

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# ============================================================
# CONFIG (lock-in)
# ============================================================
BASE_DIR = "/Users/joshmacbook/python_projects/OAD/Data/in Use"
ROUNDS_PATH = f"{BASE_DIR}/combined_rounds_all_2017_2026.csv"
ODDS_XLSX   = f"{BASE_DIR}/Odds_and_Results.xlsx"
SCHED_2024  = f"{BASE_DIR}/OAD_2024.xlsx"
SCHED_2025  = f"{BASE_DIR}/OAD_2025.xlsx"
SCHED_2026  = f"{BASE_DIR}/OAD_2026.xlsx"

WINDOWS = (40, 24, 12)
LR_C = 0.05
LR_MAX_ITER = 2000
ODDS_SCALE = 0.15

TOP_GROUP_N = 10  # for analysis summaries

# ============================================================
# Helpers
# ============================================================
def _logit(p, eps=1e-8):
    p = np.clip(np.asarray(p, dtype=float), eps, 1 - eps)
    return np.log(p / (1 - p))

def fit_lr(train_df: pd.DataFrame, feat_cols, label_col="y_win"):
    tr = train_df.dropna(subset=feat_cols + [label_col]).copy()
    ytr = tr[label_col].astype(int).values
    if np.unique(ytr).size < 2:
        return None
    m = LogisticRegression(C=LR_C, max_iter=LR_MAX_ITER, solver="lbfgs")
    m.fit(tr[feat_cols].values, ytr)
    return m

def merge_asof_by_group(left: pd.DataFrame, right: pd.DataFrame,
                        by: str, left_on: str, right_on: str,
                        direction="backward", allow_exact_matches=True) -> pd.DataFrame:
    """
    Avoids pandas 'left keys must be sorted' global constraint by running merge_asof per group.
    CRITICAL: we explicitly restore `by` as a column, so dg_id can't disappear.
    """
    out = []
    right_groups = {k: g.sort_values(right_on) for k, g in right.groupby(by, sort=False)}

    for k, lgrp in left.groupby(by, sort=False):
        rgrp = right_groups.get(k)
        if rgrp is None or rgrp.empty:
            continue

        lgrp2 = lgrp.sort_values(left_on).copy()

        merged = pd.merge_asof(
            lgrp2,
            rgrp.sort_values(right_on),
            left_on=left_on,
            right_on=right_on,
            direction=direction,
            allow_exact_matches=allow_exact_matches,
        )
        merged[by] = k
        out.append(merged)

    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

def build_player_rolling_features(rounds_df: pd.DataFrame, windows=(40,24,12)) -> pd.DataFrame:
    df = rounds_df[["dg_id","round_date","sg_total"]].copy()
    df = df.sort_values(["dg_id","round_date"]).reset_index(drop=True)
    g = df.groupby("dg_id", group_keys=False)

    df["n_rounds_to_date"] = g.cumcount() + 1

    for w in windows:
        df[f"sg_total_L{w}"] = (
            g["sg_total"]
            .rolling(window=w, min_periods=min(10, w))
            .mean()
            .reset_index(level=0, drop=True)
        )
    return df

def analyze_event_board(board: pd.DataFrame, top_group_n=10) -> pd.Series:
    b = board.copy()

    out = {}
    out["n_field"] = int(len(b))
    out["p_sum"] = float(b["p_win_share"].sum())

    # concentration / dominance
    out["top10_p_sum"] = float(b.head(10)["p_win_share"].sum())
    out["top20_p_sum"] = float(b.head(20)["p_win_share"].sum())
    out["top5_p_sum"]  = float(b.head(5)["p_win_share"].sum())

    # odds agreement (rank corr): lower close_odds is "better" so rank ascending
    if "close_odds" in b.columns and b["close_odds"].notna().any():
        tmp = b.dropna(subset=["close_odds","p_win_share"]).copy()
        if len(tmp) >= 5:
            tmp["rank_model"] = tmp["p_win_share"].rank(ascending=False, method="average")
            tmp["rank_odds"]  = tmp["close_odds"].rank(ascending=True, method="average")
            out["corr_rank_model_vs_odds"] = float(tmp["rank_model"].corr(tmp["rank_odds"]))
        else:
            out["corr_rank_model_vs_odds"] = np.nan
    else:
        out["corr_rank_model_vs_odds"] = np.nan

    # top group diagnostics
    tg = b.head(top_group_n).copy()
    out["top_group_n"] = int(len(tg))
    out["top_group_mean_sgL40"] = float(tg["sg_total_L40"].mean()) if "sg_total_L40" in tg else np.nan
    out["top_group_mean_sgL24"] = float(tg["sg_total_L24"].mean()) if "sg_total_L24" in tg else np.nan
    out["top_group_mean_sgL12"] = float(tg["sg_total_L12"].mean()) if "sg_total_L12" in tg else np.nan
    out["top_group_mean_days_since"] = float(tg["days_since_last_round"].mean()) if "days_since_last_round" in tg else np.nan
    out["top_group_mean_close_odds"] = float(tg["close_odds"].mean()) if "close_odds" in tg else np.nan

    return pd.Series(out)

# ============================================================
# 1) LOAD SCHEDULES -> sched_all
# ============================================================
sched_24 = pd.read_excel(SCHED_2024)
sched_25 = pd.read_excel(SCHED_2025)
sched_26 = pd.read_excel(SCHED_2026)
sched_all_raw = pd.concat([sched_24, sched_25, sched_26], ignore_index=True)

# Find date + name columns
date_col = next((c for c in ["start_date","event_start_date","date","event_date"] if c in sched_all_raw.columns), None)
name_col = next((c for c in ["event_name","tournament","name"] if c in sched_all_raw.columns), None)
if date_col is None:
    raise ValueError("Schedule missing start date column (start_date / event_start_date / date / event_date).")
if name_col is None:
    raise ValueError("Schedule missing event name column (event_name / tournament / name).")

sched_all = sched_all_raw.copy()
sched_all["year"] = pd.to_numeric(sched_all["year"], errors="coerce")
sched_all["event_id"] = pd.to_numeric(sched_all["event_id"], errors="coerce")
sched_all[date_col] = pd.to_datetime(sched_all[date_col], errors="coerce")
sched_all[name_col] = sched_all[name_col].astype(str)

sched_all = sched_all.dropna(subset=["year","event_id",date_col]).copy()
sched_all["year"] = sched_all["year"].astype(int)
sched_all["event_id"] = sched_all["event_id"].astype(int)
sched_all = sched_all.rename(columns={date_col:"event_start_date", name_col:"event_name"})
sched_all = sched_all[["year","event_id","event_start_date","event_name"]].copy()

sched_2026_only = sched_all[sched_all["year"] == 2026].sort_values("event_start_date").reset_index(drop=True)
if len(sched_2026_only) < 3:
    raise ValueError("Need at least 2 events in 2026 schedule to run Week 1 and Week 2.")

week1 = sched_2026_only.iloc[0]
week2 = sched_2026_only.iloc[1]
week3 = sched_2026_only.iloc[2]

# ============================================================
# 2) LOAD ODDS/RESULTS
# ============================================================
odds = pd.read_excel(ODDS_XLSX)

need_odds = {"year","event_id","dg_id","close_odds","finish_num"}
missing = need_odds - set(odds.columns)
if missing:
    raise ValueError(f"Odds_and_Results.xlsx missing required columns: {sorted(missing)}")

for c in ["year","event_id","dg_id"]:
    odds[c] = pd.to_numeric(odds[c], errors="coerce")
odds["close_odds"] = pd.to_numeric(odds["close_odds"], errors="coerce")
odds["finish_num"] = pd.to_numeric(odds["finish_num"], errors="coerce")

odds = odds.dropna(subset=["year","event_id","dg_id"]).copy()
odds["year"] = odds["year"].astype(int)
odds["event_id"] = odds["event_id"].astype(int)
odds["dg_id"] = odds["dg_id"].astype(int)

# Restrict odds to schedule event_ids
oad_event_ids = sorted(sched_all["event_id"].unique().tolist())
odds = odds[odds["event_id"].isin(oad_event_ids)].copy()

# ============================================================
# 3) LOAD ROUNDS + BUILD ROLLING FEATURES
# ============================================================
rounds = pd.read_csv(ROUNDS_PATH)

need_rounds = {"round_date","dg_id","player_name","sg_total"}
missing = need_rounds - set(rounds.columns)
if missing:
    raise ValueError(f"combined_rounds file missing required columns: {sorted(missing)}")

rounds["round_date"] = pd.to_datetime(rounds["round_date"], errors="coerce")
rounds["dg_id"] = pd.to_numeric(rounds["dg_id"], errors="coerce")
rounds["sg_total"] = pd.to_numeric(rounds["sg_total"], errors="coerce")
rounds = rounds.dropna(subset=["round_date","dg_id","sg_total"]).copy()
rounds["dg_id"] = rounds["dg_id"].astype(int)
rounds["player_name"] = rounds["player_name"].astype(str)
rounds = rounds.sort_values(["dg_id","round_date"]).reset_index(drop=True)

name_map = (
    rounds.sort_values(["dg_id","round_date"])
          .groupby("dg_id")["player_name"]
          .last()
          .to_dict()
)

rounds_roll = build_player_rolling_features(rounds, windows=WINDOWS)

# ============================================================
# 4) BUILD X (player-event rows + features)
# ============================================================
evs = sched_all.copy()
evs["cutoff_date"] = evs["event_start_date"] - pd.Timedelta(days=1)

pe = odds.merge(
    evs[["year","event_id","event_start_date","cutoff_date","event_name"]],
    on=["year","event_id"],
    how="inner",
).copy()

pe["player_name"] = pe["dg_id"].map(name_map).fillna(pe["dg_id"].apply(lambda x: f"dg_{x}"))

# y_win known only where finish_num is present
pe["y_win"] = np.where(pe["finish_num"].notna(), (pe["finish_num"] == 1).astype(int), np.nan)

left = pe.dropna(subset=["dg_id","cutoff_date"]).copy()
left["cutoff_date"] = pd.to_datetime(left["cutoff_date"], errors="coerce")
left = left.dropna(subset=["cutoff_date"]).copy()
left["dg_id"] = left["dg_id"].astype(int)

right = rounds_roll.dropna(subset=["dg_id","round_date"]).copy()
right["dg_id"] = right["dg_id"].astype(int)
right["round_date"] = pd.to_datetime(right["round_date"], errors="coerce")
right = right.dropna(subset=["round_date"]).copy()

X = merge_asof_by_group(
    left=left,
    right=right,
    by="dg_id",
    left_on="cutoff_date",
    right_on="round_date",
    direction="backward",
    allow_exact_matches=True,
)

if X.empty:
    raise ValueError("As-of merge produced empty X. Check dg_id overlap between odds field and rounds file.")

# ensure dg_id exists (guard against any weirdness)
if "dg_id" not in X.columns:
    raise ValueError("dg_id missing from X after merge_asof_by_group. Something is very wrong with group merge.")

X = X.rename(columns={"round_date":"last_round_date"})

X["days_since_last_round"] = (pd.to_datetime(X["cutoff_date"]) - pd.to_datetime(X["last_round_date"])).dt.days
X["days_since_last_round"] = pd.to_numeric(X["days_since_last_round"], errors="coerce").clip(lower=0)
X["days_since_last_round_log1p"] = np.log1p(X["days_since_last_round"].fillna(0.0))

X["p_odds_raw"] = 1.0 / pd.to_numeric(X["close_odds"], errors="coerce")
X["p_odds_raw"] = X["p_odds_raw"].replace([np.inf, -np.inf], np.nan)
X["p_odds_raw"] = X["p_odds_raw"].fillna(X["p_odds_raw"].median())

X["p_odds_share"] = X["p_odds_raw"] / (X.groupby(["year","event_id"])["p_odds_raw"].transform("sum") + 1e-12)
X["odds_logit_scaled"] = ODDS_SCALE * _logit(X["p_odds_share"].values)

BASE_FEATS = [f"sg_total_L{w}" for w in WINDOWS] + ["odds_logit_scaled"]
LAYOFF_FEATS = ["days_since_last_round_log1p"]
FEATS = BASE_FEATS + LAYOFF_FEATS

X = X.dropna(subset=[f"sg_total_L{w}" for w in WINDOWS] + ["days_since_last_round_log1p","odds_logit_scaled"]).copy()

print("X rows:", len(X), "| years in X:", sorted(X["year"].unique().tolist())[:10], "...")

# ============================================================
# Scoring function for a single 2026 event row
def score_2026_event(ev_row: pd.Series):
    ev_id = int(ev_row["event_id"])
    ev_date = pd.to_datetime(ev_row["event_start_date"])
    ev_name = str(ev_row["event_name"])

    # train on all known outcomes strictly before this event date
    train_df = X[(X["y_win"].notna()) & (pd.to_datetime(X["event_start_date"]) < ev_date)].copy()
    test_df  = X[(X["year"] == 2026) & (X["event_id"] == ev_id)].copy()

    if test_df.empty:
        raise ValueError(f"No rows found for 2026 event_id={ev_id} in Odds_and_Results field.")
    if train_df.empty:
        raise ValueError("Train df empty. Do you have finish_num populated for prior years in Odds_and_Results.xlsx?")

    # --- HARDEN: ensure expected event columns exist ---
    if "event_start_date" not in test_df.columns:
        test_df["event_start_date"] = ev_date
    else:
        test_df["event_start_date"] = pd.to_datetime(test_df["event_start_date"], errors="coerce").fillna(ev_date)

    if "event_name" not in test_df.columns:
        # if merge produced suffixed cols, pull one; otherwise just stamp from schedule row
        for cand in ["event_name_y", "event_name_x", "event_name_sched"]:
            if cand in test_df.columns:
                test_df["event_name"] = test_df[cand].astype(str)
                break
        if "event_name" not in test_df.columns:
            test_df["event_name"] = ev_name
    else:
        test_df["event_name"] = test_df["event_name"].astype(str).fillna(ev_name)

    model = fit_lr(train_df, FEATS, label_col="y_win")
    if model is None:
        raise ValueError("Training window has no class variation (no wins). Check your training filter/data.")

    # predict
    p = model.predict_proba(test_df[FEATS].values)[:, 1]
    test_df = test_df.copy()
    test_df["p_win_raw"] = p
    test_df["p_win_share"] = test_df["p_win_raw"] / (test_df["p_win_raw"].sum() + 1e-12)

    # --- HARDEN: build board using only existing cols, but guarantee the must-haves ---
    must_cols = ["year","event_id","event_start_date","event_name","dg_id","player_name",
                 "p_win_raw","p_win_share","close_odds","p_odds_share",
                 "sg_total_L40","sg_total_L24","sg_total_L12","days_since_last_round", "finish_num"]
    for c in must_cols:
        if c not in test_df.columns:
            # create missing optional columns as NaN (shouldn't happen for dg_id/player_name, but protects you)
            test_df[c] = np.nan

    # attach actual finish if present in odds file (works even if missing -> NaN)
    finish_lookup = odds[["year","event_id","dg_id","finish_num"]].copy()
    finish_lookup["finish_num"] = pd.to_numeric(finish_lookup["finish_num"], errors="coerce")

    test_df = test_df.merge(
        finish_lookup,
        on=["year","event_id","dg_id"],
        how="left",
        suffixes=("", "_actual"),
    )

    # if a merge created finish_num_actual, prefer it
    if "finish_num_actual" in test_df.columns:
        test_df["finish_num"] = test_df["finish_num_actual"].fillna(test_df.get("finish_num"))

    board = test_df[must_cols].copy()
    board = board.sort_values("p_win_share", ascending=False).reset_index(drop=True)
    board["pred_rank"] = np.arange(1, len(board) + 1)

    top10 = board.head(10).copy()
    top10["p_win_pct"] = (100 * top10["p_win_share"]).round(2)
    top10["p_odds_pct"] = (100 * top10["p_odds_share"]).round(2)

    summary = analyze_event_board(board, top_group_n=TOP_GROUP_N)

    return board, top10, summary


# ============================================================
# RUN WEEK 1 + WEEK 2
# ============================================================
print(f"\n2026 Week 1: event_id={int(week1['event_id'])} | {pd.to_datetime(week1['event_start_date']).date()} | {week1['event_name']}")
board_w1, top10_w1, summary_w1 = score_2026_event(week1)
display(summary_w1.to_frame("week1").T)
display(top10_w1[[
    "pred_rank","player_name","dg_id",
    "p_win_pct","p_odds_pct","close_odds",
    "sg_total_L40","sg_total_L24","sg_total_L12",
    "days_since_last_round", "finish_num"
]])

print(f"\n2026 Week 2: event_id={int(week2['event_id'])} | {pd.to_datetime(week2['event_start_date']).date()} | {week2['event_name']}")
board_w2, top10_w2, summary_w2 = score_2026_event(week2)
display(summary_w2.to_frame("week2").T)
display(top10_w2[[
    "pred_rank","player_name","dg_id",
    "p_win_pct","p_odds_pct","close_odds",
    "sg_total_L40","sg_total_L24","sg_total_L12",
    "days_since_last_round", "finish_num"
]])

print(f"\n2026 Week 3: event_id={int(week3['event_id'])} | {pd.to_datetime(week3['event_start_date']).date()} | {week3['event_name']}")
board_w3, top10_w3, summary_w3 = score_2026_event(week3)
display(summary_w3.to_frame("week3").T)
display(top10_w3[[
    "pred_rank","player_name","dg_id",
    "p_win_pct","p_odds_pct","close_odds",
    "sg_total_L40","sg_total_L24","sg_total_L12",
    "days_since_last_round", "finish_num"
]])

# ============================================================
# SAVE OUTPUTS
# ============================================================
OUT_W1 = f"{BASE_DIR}/win_model_2026_week1_board.csv"
OUT_W2 = f"{BASE_DIR}/win_model_2026_week2_board.csv"
OUT_W1_T10 = f"{BASE_DIR}/win_model_2026_week1_top10.csv"
OUT_W2_T10 = f"{BASE_DIR}/win_model_2026_week2_top10.csv"
OUT_W3 = f"{BASE_DIR}/win_model_2026_week3_board.csv"
OUT_W3_T10 = f"{BASE_DIR}/win_model_2026_week3_top10.csv"

board_w1.to_csv(OUT_W1, index=False)
board_w2.to_csv(OUT_W2, index=False)
top10_w1.to_csv(OUT_W1_T10, index=False)
top10_w2.to_csv(OUT_W2_T10, index=False)
board_w3.to_csv(OUT_W3, index=False)
top10_w3.to_csv(OUT_W3_T10, index=False)


print("\nSaved:")
print(" -", OUT_W1)
print(" -", OUT_W2)
print(" -", OUT_W1_T10)
print(" -", OUT_W2_T10)
print(" -", OUT_W3)
print(" -", OUT_W3_T10)

/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_1658/834137470.py:182: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  rounds = pd.read_csv(ROUNDS_PATH)


X rows: 7770 | years in X: [2024, 2025, 2026] ...

2026 Week 1: event_id=6 | 2026-01-15 | Sony Open in Hawaii


,n_field,p_sum,top10_p_sum,top20_p_sum,top5_p_sum,corr_rank_model_vs_odds,top_group_n,top_group_mean_sgL40,top_group_mean_sgL24,top_group_mean_sgL12,top_group_mean_days_since,top_group_mean_close_odds
week1,109.0,1.0,0.364305,0.539927,0.246735,0.511719,10.0,1.110192,1.374987,1.7907,60.4,45.9


,pred_rank,player_name,dg_id,p_win_pct,p_odds_pct,close_odds,sg_total_L40,sg_total_L24,sg_total_L12,days_since_last_round,finish_num
0,1,"Griffin, Ben",24968,10.52,4.70,14.0,1.771675,2.017500,2.912667,66,19.0
1,2,"Spaun, J.J.",17536,5.90,3.29,20.0,1.471125,1.567458,2.429250,38,40.0
2,3,"McGreevy, Max",23505,2.81,0.59,111.0,0.781725,1.146083,1.892833,52,NaN
3,4,"Hoey, Rico",23504,2.72,1.43,46.0,1.170500,1.543667,1.476167,52,50.0
4,5,"Whaley, Vince",25908,2.72,0.72,91.0,0.825075,1.236542,1.776417,52,40.0
5,6,"Henley, Russell",14578,2.67,5.48,12.0,1.194300,1.663292,1.346417,122,19.0
6,7,"Kim, Si Woo",14609,2.41,3.29,20.0,0.869900,1.363000,1.495083,52,11.0
7,8,"Matsuyama, Hideki",13562,2.28,3.66,18.0,1.061225,0.655917,1.766417,38,13.0
8,9,"McCarty, Matt",28635,2.22,1.00,66.0,0.948875,1.074458,1.545083,80,55.0
9,10,"Meissner, Mac",28159,2.18,1.08,61.0,1.007525,1.481958,1.266667,52,40.0



2026 Week 2: event_id=2 | 2026-01-22 | The American Express


...

,pred_rank,player_name,dg_id,p_win_pct,p_odds_pct,close_odds,sg_total_L40,sg_total_L24,sg_total_L12,days_since_last_round,finish_num
0,1,"Scheffler, Scottie",18417,10.74,18.92,3.4,2.994050,3.110625,2.429083,45,1.0
1,2,"Valimaki, Sami",23816,4.27,0.49,131.0,1.162050,1.420208,2.726167,59,NaN
2,3,"Griffin, Ben",24968,4.08,3.39,19.0,1.794350,1.991583,2.053417,3,24.0
3,4,"Fitzpatrick, Matt",17646,3.30,2.07,31.0,1.931775,2.178542,1.584917,66,63.0
4,5,"Kim, Si Woo",14609,2.16,1.79,36.0,1.063525,1.462083,1.719417,3,6.0
5,6,"Noren, Alex",10419,2.00,1.40,46.0,1.707675,1.378583,1.417583,45,NaN
6,7,"Hojgaard, Rasmus",23838,1.91,1.40,46.0,1.618600,1.547125,1.290833,66,44.0
7,8,"Henley, Russell",14578,1.91,2.80,23.0,1.399050,1.409417,1.443333,3,8.0
8,9,"McGreevy, Max",23505,1.75,0.21,301.0,0.759025,1.270958,1.681833,5,27.0
9,10,"Fowler, Rickie",12965,1.73,1.26,51.0,1.093225,1.503083,1.365417,157,18.0



2026 Week 3: event_id=4 | 2026-01-29 | Farmers Insurance Open


,n_field,p_sum,top10_p_sum,top20_p_sum,top5_p_sum,corr_rank_model_vs_odds,top_group_n,top_group_mean_sgL40,top_group_mean_sgL24,top_group_mean_sgL12,top_group_mean_days_since,top_group_mean_close_odds
week3,145.0,1.0,0.309489,0.469063,0.202565,0.683745,10.0,1.264598,1.528083,1.727058,33.1,43.43


,pred_rank,player_name,dg_id,p_win_pct,p_odds_pct,close_odds,sg_total_L40,sg_total_L24,sg_total_L12,days_since_last_round,finish_num
0,1,"Higgo, Garrick",24342,8.50,1.24,58.0,1.028000,2.078625,3.120750,80,NaN
1,2,"Kim, Si Woo",14609,3.57,2.66,27.0,1.232250,1.755708,2.005833,3,NaN
2,3,"Young, Cameron",26651,3.07,3.68,19.5,1.591925,2.063917,1.345750,52,NaN
3,4,"McCarty, Matt",28635,2.77,1.05,68.0,1.247700,1.635000,1.724583,3,NaN
4,5,"Olesen, Thorbjorn",13412,2.34,1.09,66.0,1.610300,1.665625,1.313583,3,NaN
5,6,"Matsuyama, Hideki",13562,2.24,2.31,31.0,0.989875,0.912708,1.907417,10,NaN
6,7,"Rodgers, Patrick",16283,2.20,1.26,57.0,0.552000,1.479917,1.761500,3,NaN
7,8,"Spaun, J.J.",17536,2.13,2.47,29.0,1.309950,1.124875,1.570000,10,NaN
8,9,"Penge, Marco",22465,2.08,1.14,63.0,1.791975,1.355667,1.156167,59,NaN
9,10,"Schauffele, Xander",19895,2.04,4.54,15.8,1.292000,1.208792,1.365000,108,NaN



Saved:
 - /Users/joshmacbook/python_projects/OAD/Data/in Use/win_model_2026_week1_board.csv
 - /Users/joshmacbook/python_projects/OAD/Data/in Use/win_model_2026_week2_board.csv
 - /Users/joshmacbook/python_projects/OAD/Data/in Use/win_model_2026_week1_top10.csv
 - /Users/joshmacbook/python_projects/OAD/Data/in Use/win_model_2026_week2_top10.csv
 - /Users/joshmacbook/python_projects/OAD/Data/in Use/win_model_2026_week3_board.csv
 - /Users/joshmacbook/python_projects/OAD/Data/in Use/win_model_2026_week3_top10.csv
